# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishnu-varthan1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:

from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError("HF_TOKEN is missing. Check Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE huggingface, TOKEN ?)",
    [hf_token]
)

print("Hugging Face token loaded and DuckDB connection initialized.")


Hugging Face token loaded and DuckDB connection initialized.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis:** One row represents one report date, one pseudonymized client, and one pseudonymized content item in `fact_content_daily_performance`.

**Time window:** March 2026 (`2026-03`) is the development window. June 2026 is reserved as a sealed test month.

**Purpose:** Explore historical content performance to support prioritizing pages for review. Verify the actual grain and date range using warehouse queries before relying on these assumptions.


In [2]:

import duckdb
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_secret "
    "(TYPE huggingface, TOKEN ?)",
    [hf_token]
)

rel = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

# Inspect the real schema before choosing columns.
schema_df = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{rel}')"
).fetchdf()

display(schema_df)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features:** Historical impressions, clicks, sessions, content age, and days since the last update, where available in the actual schema.

**Label/proxy:** A future content-performance outcome, to be defined only after confirming the available fields and time windows. Do not use a future outcome as an input feature.

**Context:** Pseudonymized content and client keys, report date, and month. These help define the analysis grain and time window.

**Excluded:** Raw client-identifying information, URLs, private queries, and any field that could reveal client identity. Exclude future outcome fields from model inputs to prevent leakage.


In [4]:

# Define the fields used in this analysis.
columns = schema_df["column_name"].tolist()

feature_candidates = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "sessions_organic",
]

context_fields = [
    "report_date",
    "month",
    "client_hash_id",
    "content_hash_id",
]

availability_fields = [
    "gsc_data_available",
    "ga4_data_available",
    "client_has_gsc",
    "client_has_ga4",
]

print("Available features:")
print([c for c in feature_candidates if c in columns])

print("\nAvailable context fields:")
print([c for c in context_fields if c in columns])

print("\nAvailability fields:")
print([c for c in availability_fields if c in columns])

print("\nExcluded from model inputs:")
print("Client/content hash IDs (identifiers), future outcome fields,")
print("and any client-identifying information.")


Available features:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'sessions_organic']

Available context fields:
['report_date', 'month', 'client_hash_id', 'content_hash_id']

Availability fields:
['gsc_data_available', 'ga4_data_available', 'client_has_gsc', 'client_has_ga4']

Excluded from model inputs:
Client/content hash IDs (identifiers), future outcome fields,
and any client-identifying information.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:

# Print column names and likely date, key, and availability fields.
columns = schema_df["column_name"].tolist()

print("All columns:")
print(columns)

print("\nPotential date/month fields:")
print([c for c in columns if "date" in c.lower() or "month" in c.lower()])

print("\nPotential key fields:")
print([c for c in columns if any(
    term in c.lower()
    for term in ["client", "content", "key", "id"]
)])

print("\nPotential availability fields:")
print([c for c in columns if any(
    term in c.lower()
    for term in ["available", "availability", "is_", "valid"]
)])


All columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

Potential date/month fields:
['report_date', 'month']

Potential key fields:
['client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'sessions_paid']

Potential availability fields:
['gsc_data_available', 'ga4_data_available']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


The warehouse is an unbalanced panel, so different pseudonymized clients may have different history lengths. Earlier periods may contain Search Console data without equivalent Analytics history. Therefore, comparisons across clients and time periods may not be directly comparable.

Overlapping historical and future windows can cause target leakage if future outcomes are included in model inputs. The analysis is observational and cannot establish that updating a page causes improved performance. Findings should be treated as directional evidence for decision support, not proof of causation.


In [5]:

# Five candidate features for March 2026.
feature_sql = f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_sessions,
        sessions_organic
    FROM read_parquet('{rel}')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
"""

feature_df = con.execute(feature_sql).fetchdf()

print("Feature frame shape:", feature_df.shape)
display(feature_df.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (3611061, 8)


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,sessions_organic
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,<NA>,<NA>
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,<NA>,<NA>
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,<NA>,<NA>
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,<NA>,<NA>
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,<NA>,<NA>
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,<NA>,<NA>


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.